In [1]:
import pandas as pd
from pprint import pprint
import random


In [2]:
# Constants
ALLOWED_NUMBER_OF_COURSES = 3

# PreReqs
prereq_dict = {
    "DATM507": ["ITOP501"],
    "DATM508": ["ITOP501"],
    "ITOP601": ["ITOP501"],
    "PROG602": ["DATM507"],
    "DATM603": ["DATM506", "DATM507"],
    "DATM604": ["PROG504", "DATM506"],
    "DATM605": ["PROG504"],
    "SYSP606": ["ITOP501"],
    "ITMG607": ["DATM508", "ITOP601"],
    "PROF608": ["SYSP606"],
    "PRMG701": ["PROF502", "PROG602", "SYSP606"],
    "HCEV702": ["PROF505", "SYSP606", "PROG602"],
    "DATM703": ["DATM603", "DATM604", "ITMG607"],
    "NTAS704": ["SCTY503", "ITMG607"],
    "PROF710": ["PROF709"],
    "ITOP721": ["DATM604"],
    "PROG722": ["PROG504", "DATM605"],
    "ITOP723": ["ITMG607"],
    "IRMG724": ["PROF505", "PROF608", "PRMG701"]
}




In [3]:
def update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo):
    dict_subject_stats = {}
    for course_id in list_courses:
        dict_subject_stats[course_id] = df_with_prereqinfo[course_id].value_counts().to_dict()
        tot_eligible_students = 0
        for iter_key in keys_to_visit_for_elgible_students:
            if iter_key in dict_subject_stats[course_id]:
                tot_eligible_students += dict_subject_stats[course_id][iter_key]
        # Add back to dictionary
        dict_subject_stats[course_id]['tot_eligible_students'] = tot_eligible_students

    return dict_subject_stats

def func_filter_eligible_students(list_eligible_students, dict_trackCourseAssignments):
    filtered_list = []
    for student_id in list_eligible_students:
        if dict_trackCourseAssignments[student_id]["freeze_assignment"] is False:
            filtered_list.append(student_id)
    return filtered_list

# npr: No PreReq Required (no pre-req required for this course)
# pum: Prereq unmet (Prereq not passed)
# met: Prereq met 
def check_course_eligibility(course_id, prereq_dict, dict_student):
    if course_id in prereq_dict:
        eligible_course_list = prereq_dict[course_id]
        for iter_eligible_course in eligible_course_list:
            status = dict_student[iter_eligible_course]
            if status != "p":
                return "pum"
        return "met"
    else:
        return "npr"

def mark_eligible_courses(df_dict, prereq_dict):
    df_dict_with_prereq_status = {}
    for iter_student in df_dict:
        print(f"Student-ID: {iter_student}")
        dict_student = df_dict[iter_student]
        df_dict_with_prereq_status[iter_student] = {}
        for iter_sub in dict_student:
            prereq_status = "already_passed" if dict_student[iter_sub] == "p" else check_course_eligibility(iter_sub, prereq_dict, dict_student)
            print(f"iter_student: {iter_student}, iter_sub:{iter_sub}, status:{dict_student[iter_sub]}, prereq_status: {prereq_status}")
            # Fill the dictionary
            df_dict_with_prereq_status[iter_student][iter_sub] = prereq_status
    return df_dict_with_prereq_status


def func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected):
    # Filter df
    df_filtered = df_with_prereqinfo.loc[(df_with_prereqinfo[course_id] == "npr") | (df_with_prereqinfo[course_id] == "met")][course_id]
    list_eligible_students = df_filtered.index.to_list()
    list_eligible_students = func_filter_eligible_students(list_eligible_students, dict_trackCourseAssignments)
    
    if len(list_eligible_students) == 0:
        print("Eligible students not found")
    elif no_of_students_selected > len(list_eligible_students): 
        print(f"Can not select {no_of_students_selected} from {len(list_eligible_students)} eligible students. Reduce your selection.")
    else:
        list_sampled_students = random.sample(list_eligible_students, no_of_students_selected)
        print(list_sampled_students, len(list_sampled_students))
        # Modify the entries of df_with_prereqinfo
        for iter_eligible_studentid in list_sampled_students:
            df_with_prereqinfo.at[iter_eligible_studentid, course_id] = "assigned"
            # Update df_dict_with_prereq_status and dict_trackCourseAssignments 
            if dict_trackCourseAssignments[iter_eligible_studentid]["freeze_assignment"] is False:
                df_dict_with_prereq_status[iter_eligible_studentid][course_id] = "assigned"
                dict_trackCourseAssignments[iter_eligible_studentid]["assigned"] += 1
                dict_trackCourseAssignments[iter_eligible_studentid]["limit"] = ALLOWED_NUMBER_OF_COURSES


                if dict_trackCourseAssignments[iter_eligible_studentid]["assigned"] == dict_trackCourseAssignments[iter_eligible_studentid]["limit"]:
                    dict_trackCourseAssignments[iter_eligible_studentid]["freeze_assignment"] = True
                    # Lock courses
                    list_courses = df_with_prereqinfo.columns.to_list()
                    for iter_course_ids in list_courses:
                        # Lock the entry
                        df_with_prereqinfo.at[iter_eligible_studentid, iter_course_ids] = df_with_prereqinfo.at[iter_eligible_studentid, iter_course_ids] + "-LOCKED"


    return df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status
    

def print_eligibility(dict_subject_stats, list_courses):
    key_to_visit = "tot_eligible_students"
    for course_id in list_courses:
        print(f"Eligible students in {course_id}: {dict_subject_stats[course_id][key_to_visit]}")

In [4]:
df = pd.read_csv("mastersheet_trimmed.csv")
df = df.fillna("x")


In [5]:
df = df.set_index('Studentid')
df_dict = df.to_dict(orient='index')

In [6]:
df_dict.keys()

dict_keys(['shea24001', 'shea24003', 'shea24004', 'shea24005', 'shea24006', 'shea24007', 'shea24008', 'shea24009', 'shea24010', 'shea24011', 'shea24015', 'shea24017', 'shea24018', 'shea24019', 'shea24020', 'shea24022', 'shea24023', 'shea24024', 'shea24025', 'shea24026', 'shea24027', 'shea24028', 'shea24029', 'shea24030', 'shea24031', 'shea24032', 'shea24033', 'shea24034', 'shea24037', 'shea24039', 'shea24042', 'shea24043', 'shea24044', 'shea24046', 'shea24047', 'shea24048', 'shea24049', 'shea24050', 'shea24051', 'shea24054', 'shea24055', 'shea24056', 'shea24057', 'shea24058', 'shea24059', 'shea24060', 'shea24061', 'shea24062', 'shea24063', 'shea24064', 'shea24065', 'shea24066', 'shea24067', 'shea24068', 'shea24069', 'shea24070', 'shea24071', 'shea24072', 'shea24074', 'shea24075', 'shea24076', 'shea24078', 'shea24079', 'shea24080', 'shea24081', 'shea24082', 'shea24083', 'shea24084', 'shea24085', 'shea24086', 'shea24088', 'shea24089', 'shea24090', 'shea24091', 'shea24093', 'shea24094', '

In [7]:
df_dict_with_prereq_status = mark_eligible_courses(df_dict, prereq_dict)

Student-ID: shea24001
iter_student: shea24001, iter_sub:ITOP501, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:PROF502, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:SCTY503, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:PROG504, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:PROF505, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:DATM506, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:DATM507, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:DATM508, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:ITOP601, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:PROG602, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:DATM603, status:p, prereq_status: already_passed
iter_student: shea24001, iter_sub:DATM604, status:x, prereq_statu

In [8]:
df_with_prereqinfo = pd.DataFrame.from_dict(df_dict_with_prereq_status, orient='index')

In [9]:
df_with_prereqinfo

,ITOP501,PROF502,SCTY503,PROG504,PROF505,DATM506,DATM507,DATM508,ITOP601,PROG602,...,PRMG701,HCEV702,DATM703,NTAS704,PROF709,PROF710,ITOP721,PROG722,ITOP723,IRMG724
shea24001,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,...,pum,pum,pum,pum,npr,pum,pum,pum,pum,pum
shea24003,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,...,pum,pum,pum,pum,npr,pum,pum,pum,pum,pum
shea24004,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,...,pum,pum,pum,pum,npr,pum,pum,pum,pum,pum
shea24005,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,...,pum,pum,pum,pum,npr,pum,pum,pum,pum,pum
shea24006,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,already_passed,...,met,met,pum,already_passed,npr,pum,met,met,pum,pum
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
shea25016,already_passed,already_passed,already_passed,already_passed,already_passed,npr,already_passed,met,met,met,...,pum,pum,pum,already_passed,npr,pum,pum,pum,pum,pum
shea25015,already_passed,already_passed,npr,already_passed,npr,npr,already_passed,already_passed,met,met,...,pum,pum,pum,pum,npr,pum,pum,pum,pum,pum
shea25017,npr,npr,npr,npr,npr,npr,pum,pum,pum,pum,...,pum,pum,pum,pum,npr,pum,pum,pum,pum,pum
shea25018,npr,npr,npr,npr,npr,npr,pum,pum,pum,pum,...,pum,pum,pum,pum,npr,pum,pum,pum,pum,pum


In [10]:
# Prepare a dictionary for all students to track how many courses are assigned to each student
dict_trackCourseAssignments = {}
for iter_student in df_dict:
    dict_trackCourseAssignments[iter_student] = {'assigned': 0, 'limit': ALLOWED_NUMBER_OF_COURSES, 'freeze_assignment': False}
print(dict_trackCourseAssignments)

{'shea24001': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24003': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24004': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24005': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24006': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24007': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24008': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24009': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24010': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24011': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24015': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24017': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24018': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24019': {'assigned': 0, 'limit': 3, 'freeze_assignment': False}, 'shea24020': {'assi

In [11]:
# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print(dict_subject_stats)


{'ITOP501': {'already_passed': 166, 'npr': 8, 'tot_eligible_students': 8}, 'PROF502': {'already_passed': 146, 'npr': 28, 'tot_eligible_students': 28}, 'SCTY503': {'already_passed': 150, 'npr': 24, 'tot_eligible_students': 24}, 'PROG504': {'already_passed': 165, 'npr': 9, 'tot_eligible_students': 9}, 'PROF505': {'already_passed': 123, 'npr': 51, 'tot_eligible_students': 51}, 'DATM506': {'already_passed': 147, 'npr': 27, 'tot_eligible_students': 27}, 'DATM507': {'already_passed': 129, 'met': 38, 'pum': 7, 'tot_eligible_students': 38}, 'DATM508': {'already_passed': 120, 'met': 47, 'pum': 7, 'tot_eligible_students': 47}, 'ITOP601': {'met': 129, 'already_passed': 37, 'pum': 8, 'tot_eligible_students': 129}, 'PROG602': {'already_passed': 79, 'met': 51, 'pum': 44, 'tot_eligible_students': 51}, 'DATM603': {'met': 78, 'pum': 50, 'already_passed': 46, 'tot_eligible_students': 78}, 'DATM604': {'met': 122, 'pum': 29, 'already_passed': 23, 'tot_eligible_students': 122}, 'DATM605': {'met': 145, 'alr

# Assigning Subjects to Eligible Students
1. Select Subject and find how many students can enrol in it (use dict_subject_stats)
2. Get list of eligible students for the above subject (get_eligible_students(str: subject_id) -> List)
3. Use a parameter and select students randomly from that list
4. Update df_with_prereqinfo and modify cell value by "assigned"


In [ ]:
course_id = 'DATM506'
no_of_students_selected = 25

df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)


['shea25019', 'shea25017', 'shea24187', 'shea24181', 'shea25013', 'shea25004', 'shea24180', 'shea24185', 'shea24183', 'shea25015', 'shea25006', 'shea24150', 'shea25003', 'shea24092', 'shea25008', 'shea25016', 'shea24101', 'shea25012', 'shea24182', 'shea25010', 'shea25007', 'shea25014', 'shea25002', 'shea24179', 'shea25009'] 25
Eligible students in ITOP501: 8
Eligible students in PROF502: 28
Eligible students in SCTY503: 24
Eligible students in PROG504: 9
Eligible students in PROF505: 51
Eligible students in DATM506: 2
Eligible students in DATM507: 38
Eligible students in DATM508: 47
Eligible students in ITOP601: 129
Eligible students in PROG602: 51
Eligible students in DATM603: 78
Eligible students in DATM604: 122
Eligible students in DATM605: 145
Eligible students in SYSP606: 81
Eligible students in ITMG607: 23
Eligible students in PROF608: 56
Eligible students in PRMG701: 6
Eligible students in HCEV702: 45
Eligible students in DATM703: 13
Eligible students in NTAS704: 1
Eligible stud

In [13]:
df_with_prereqinfo.to_csv("check.csv")

In [14]:
course_id = 'ITOP601'
no_of_students_selected = 25
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)


['shea24154', 'shea25015', 'shea24168', 'shea24075', 'shea24087', 'shea24140', 'shea24155', 'shea24133', 'shea24187', 'shea25006', 'shea24120', 'shea24114', 'shea24156', 'shea24147', 'shea25014', 'shea24112', 'shea24108', 'shea24149', 'shea24170', 'shea24062', 'shea24151', 'shea25016', 'shea24174', 'shea24135', 'shea24024'] 25
Eligible students in ITOP501: 8
Eligible students in PROF502: 28
Eligible students in SCTY503: 24
Eligible students in PROG504: 9
Eligible students in PROF505: 51
Eligible students in DATM506: 2
Eligible students in DATM507: 38
Eligible students in DATM508: 47
Eligible students in ITOP601: 104
Eligible students in PROG602: 51
Eligible students in DATM603: 78
Eligible students in DATM604: 122
Eligible students in DATM605: 145
Eligible students in SYSP606: 81
Eligible students in ITMG607: 23
Eligible students in PROF608: 56
Eligible students in PRMG701: 6
Eligible students in HCEV702: 45
Eligible students in DATM703: 13
Eligible students in NTAS704: 1
Eligible stud

In [15]:
df_with_prereqinfo.to_csv("check.csv")

In [16]:
course_id = 'DATM604'
no_of_students_selected = 25
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)

['shea24153', 'shea24005', 'shea24147', 'shea24125', 'shea24166', 'shea24094', 'shea24170', 'shea24082', 'shea24093', 'shea24102', 'shea24108', 'shea24044', 'shea24103', 'shea24022', 'shea24122', 'shea24009', 'shea24028', 'shea24008', 'shea24098', 'shea24027', 'shea24061', 'shea24010', 'shea24118', 'shea24001', 'shea24088'] 25
Eligible students in ITOP501: 8
Eligible students in PROF502: 28
Eligible students in SCTY503: 24
Eligible students in PROG504: 9
Eligible students in PROF505: 51
Eligible students in DATM506: 2
Eligible students in DATM507: 38
Eligible students in DATM508: 47
Eligible students in ITOP601: 104
Eligible students in PROG602: 51
Eligible students in DATM603: 78
Eligible students in DATM604: 97
Eligible students in DATM605: 145
Eligible students in SYSP606: 81
Eligible students in ITMG607: 23
Eligible students in PROF608: 56
Eligible students in PRMG701: 6
Eligible students in HCEV702: 45
Eligible students in DATM703: 13
Eligible students in NTAS704: 1
Eligible stude

In [17]:
course_id = 'DATM508'
no_of_students_selected = 47
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)

['shea25010', 'shea24087', 'shea24086', 'shea25012', 'shea24181', 'shea24100', 'shea24070', 'shea24017', 'shea24183', 'shea25016', 'shea24180', 'shea24169', 'shea24078', 'shea25002', 'shea24075', 'shea24137', 'shea24071', 'shea24124', 'shea24063', 'shea24097', 'shea24093', 'shea25014', 'shea25006', 'shea24161', 'shea25013', 'shea24136', 'shea24112', 'shea24058', 'shea25004', 'shea24185', 'shea24037', 'shea24141', 'shea24131', 'shea24156', 'shea24187', 'shea25008', 'shea24023', 'shea24064', 'shea24067', 'shea24113', 'shea24018', 'shea24090', 'shea24051', 'shea24158', 'shea24066', 'shea24061', 'shea24094'] 47
Eligible students in ITOP501: 8
Eligible students in PROF502: 28
Eligible students in SCTY503: 24
Eligible students in PROG504: 9
Eligible students in PROF505: 51
Eligible students in DATM506: 2
Eligible students in DATM507: 37
Eligible students in DATM508: 0
Eligible students in ITOP601: 104
Eligible students in PROG602: 48
Eligible students in DATM603: 78
Eligible students in DATM

In [18]:
course_id = 'DATM605'
no_of_students_selected = 141
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)

['shea25013', 'shea24131', 'shea24008', 'shea24072', 'shea24023', 'shea24145', 'shea24075', 'shea24005', 'shea25008', 'shea24180', 'shea24064', 'shea24175', 'shea24047', 'shea24155', 'shea24141', 'shea24082', 'shea25007', 'shea24061', 'shea25004', 'shea24154', 'shea24129', 'shea25015', 'shea24096', 'shea24062', 'shea24117', 'shea25012', 'shea24076', 'shea24024', 'shea24065', 'shea24092', 'shea24127', 'shea24142', 'shea24158', 'shea24097', 'shea24018', 'shea24152', 'shea24134', 'shea24010', 'shea24046', 'shea24176', 'shea24128', 'shea24151', 'shea24022', 'shea24037', 'shea24051', 'shea24083', 'shea24137', 'shea24153', 'shea24017', 'shea24124', 'shea24060', 'shea24033', 'shea24125', 'shea25009', 'shea24089', 'shea24122', 'shea24148', 'shea24028', 'shea24085', 'shea24011', 'shea24078', 'shea24009', 'shea24067', 'shea24159', 'shea24056', 'shea24100', 'shea24165', 'shea24042', 'shea24161', 'shea24136', 'shea24081', 'shea24113', 'shea24160', 'shea24001', 'shea24079', 'shea25010', 'shea24098'

In [19]:
course_id = 'ITMG607'
no_of_students_selected = 23
df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status = func_assign_course(df_with_prereqinfo, dict_trackCourseAssignments, df_dict_with_prereq_status, course_id, no_of_students_selected)

# Call the function
list_courses = df_with_prereqinfo.columns.to_list()
keys_to_visit_for_elgible_students = ["npr", "met"]
dict_subject_stats = update_dict_of_subjects(list_courses, keys_to_visit_for_elgible_students, df_with_prereqinfo)
print_eligibility(dict_subject_stats, list_courses)

['shea24015', 'shea24109', 'shea24009', 'shea24044', 'shea24010', 'shea24008', 'shea24046', 'shea24047', 'shea24033', 'shea24003', 'shea24029', 'shea24032', 'shea24006', 'shea24001', 'shea24005', 'shea24048', 'shea24027', 'shea24039', 'shea24089', 'shea24004', 'shea24050', 'shea24022', 'shea24011'] 23
Eligible students in ITOP501: 8
Eligible students in PROF502: 23
Eligible students in SCTY503: 10
Eligible students in PROG504: 9
Eligible students in PROF505: 35
Eligible students in DATM506: 2
Eligible students in DATM507: 22
Eligible students in DATM508: 0
Eligible students in ITOP601: 91
Eligible students in PROG602: 41
Eligible students in DATM603: 72
Eligible students in DATM604: 93
Eligible students in DATM605: 0
Eligible students in SYSP606: 55
Eligible students in ITMG607: 0
Eligible students in PROF608: 51
Eligible students in PRMG701: 6
Eligible students in HCEV702: 42
Eligible students in DATM703: 13
Eligible students in NTAS704: 1
Eligible students in PROF709: 141
Eligible st

In [20]:
df_with_prereqinfo.to_csv("check.csv")